## Markdowncaractertextesplitter + recursive texte splitter

ce que je dois faire :
<ul>
<li>getting the pdf in markdown format </li>
<li>diviser selon les section </li>
<li>diviser selon les paragraphes</li>
<li>ajouter a la base donnée*</li>

</ul>






In [1]:
from unstructured.partition.pdf import partition_pdf

c:\Users\tchouarr\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
elements = partition_pdf(
    filename="..\data\[GU] Calcul des besoins V12.pdf",
    strategy="auto",  # meilleur parsing --> faux hi-res est le meilleur avec tesseract
    #infer_table_structure=True,
)

No languages specified, defaulting to English.


In [4]:
len(elements)

247

In [5]:
for el in elements[:50]:
    print(type(el))
    print(el.text)
    print(el.metadata)
    print("=" * 50)

<class 'unstructured.documents.elements.Title'>
SAGE X3 – V12 – CALCUL DES BESOINS
<class 'unstructured.documents.elements.Title'>
Guide Utilisateur
<class 'unstructured.documents.elements.Text'>
Version du 22/05/2024
<class 'unstructured.documents.elements.Footer'>
1
<class 'unstructured.documents.elements.Title'>
HISTORIQUE DU DOCUMENT
<class 'unstructured.documents.elements.Title'>
Date
<class 'unstructured.documents.elements.Title'>
BZ
<class 'unstructured.documents.elements.Title'>
Chapitre
<class 'unstructured.documents.elements.Title'>
Descriptif
<class 'unstructured.documents.elements.Text'>
20/06/2011
<class 'unstructured.documents.elements.Title'>
Version initiale
<class 'unstructured.documents.elements.Text'>
19/08/2011
<class 'unstructured.documents.elements.Text'>
27/09/2018
<class 'unstructured.documents.elements.Text'>
2.1, 2.4, 2.5, 3.3
<class 'unstructured.documents.elements.ListItem'>
Possibilité de calculer le besoin selon un seuil de réappro manuel avec besoin = seu

In [14]:
elements.category.unique()

AttributeError: 'list' object has no attribute 'category'

In [15]:
from unstructured.partition.pdf import partition_pdf
from unstructured.staging.base import elements_to_markdown

ImportError: cannot import name 'elements_to_markdown' from 'unstructured.staging.base' (c:\Users\tchouarr\AppData\Local\Programs\Python\Python311\Lib\site-packages\unstructured\staging\base.py)

## Markdown

In [6]:
markdown_text = ""

for el in elements:

    page = getattr(el.metadata, "page_number", None)

    if el.category == "Title":
        markdown_text += f"\n# {el.text}\n"

    else:
        markdown_text += f"\n{el.text}\n"

In [7]:
from unstructured.staging.base import elements_to_markdown

ImportError: cannot import name 'elements_to_markdown' from 'unstructured.staging.base' (c:\Users\tchouarr\AppData\Local\Programs\Python\Python311\Lib\site-packages\unstructured\staging\base.py)

In [17]:
len(markdown_text)

17924

In [18]:
markdown = []

for el in elements:

    if not hasattr(el, "text") or not el.text:
        continue

    category = el.category
    text = el.text.strip()

    if category == "Title":
        markdown.append(f"# {text}")

    elif category == "ListItem":
        markdown.append(f"- {text}")

    elif category == "Table":
        markdown.append(f"\n## Tableau\n{text}\n")

    else:
        markdown.append(text)

markdown_text2 = "\n\n".join(markdown)

In [19]:
len(markdown_text2)

17956

In [20]:
markdown_text2

'# SAGE X3 – V12 – CALCUL DES BESOINS\n\n# Guide Utilisateur\n\nVersion du 22/05/2024\n\n1\n\n# HISTORIQUE DU DOCUMENT\n\n# Date\n\n# BZ\n\n# Chapitre\n\n# Descriptif\n\n20/06/2011\n\n# Version initiale\n\n19/08/2011\n\n27/09/2018\n\n2.1, 2.4, 2.5, 3.3\n\n- Possibilité de calculer le besoin selon un seuil de réappro manuel avec besoin = seuil réappro - Possibilité de calculer le besoin d’un site en prenant en compte les données de plusieurs sites (groupe de site). Mise à jour Guide en Version 11\n\n- Possibilité de calculer le besoin selon un seuil de réappro manuel avec besoin = seuil réappro - Possibilité de calculer le besoin d’un site en prenant en compte les données de plusieurs sites (groupe de site). Mise à jour Guide en Version 11\n\n22/05/2024\n\n# MAJ EN V12\n\nMAJ charte graphique et format\n\n2\n\n# Rédacteur\n\n# P. LEROY\n\n# J. COURIC\n\n# D. CARDANTE\n\n# B. LIFY\n\n# G. BOURRILLON\n\n# SOMMAIRE\n\n1 PARAMETRAGE ..................................................... 4 1.

In [21]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

docs = splitter.split_text(markdown_text)



In [22]:
docs

[Document(metadata={'Header 1': 'Guide Utilisateur'}, page_content='Version du 22/05/2024  \n1'),
 Document(metadata={'Header 1': 'Descriptif'}, page_content='20/06/2011'),
 Document(metadata={'Header 1': 'Version initiale'}, page_content='19/08/2011  \n27/09/2018  \n2.1, 2.4, 2.5, 3.3  \nPossibilité de calculer le besoin selon un seuil de réappro manuel avec besoin = seuil réappro - Possibilité de calculer le besoin d’un site en prenant en compte les données de plusieurs sites (groupe de site). Mise à jour Guide en Version 11  \nPossibilité de calculer le besoin selon un seuil de réappro manuel avec besoin = seuil réappro - Possibilité de calculer le besoin d’un site en prenant en compte les données de plusieurs sites (groupe de site). Mise à jour Guide en Version 11  \n22/05/2024'),
 Document(metadata={'Header 1': 'MAJ EN V12'}, page_content='MAJ charte graphique et format  \n2'),
 Document(metadata={'Header 1': 'SOMMAIRE'}, page_content='1 PARAMETRAGE ...............................

In [ ]:
from langchain_core.documents import Document



### ajout des titres dans page.content 

In [24]:

enhanced_docs = []

for doc in docs:

    header = doc.metadata.get("Header 1", "")

    content = f"""
Section: {header}

{doc.page_content}
"""

    enhanced_doc = Document(
        page_content=content,
        metadata=doc.metadata
    )

    enhanced_docs.append(enhanced_doc)

In [26]:
for doc in docs:
    print(doc.metadata
    )

{'Header 1': 'Guide Utilisateur'}
{'Header 1': 'Descriptif'}
{'Header 1': 'Version initiale'}
{'Header 1': 'MAJ EN V12'}
{'Header 1': 'SOMMAIRE'}
{'Header 1': '▪ Puis sélectionner XAP'}
{'Header 1': '2.1 TYPES DE DELAI'}
{'Header 1': 'o Dans la liste gauche sélectionner la table n°20'}
{'Header 1': '2.2 JOURS DE COMMANDE'}
{'Header 1': '➢ Données de base > Tiers > Fournisseurs'}
{'Header 1': 'Le jour de commande (facultatif)'}
{'Header 1': '➢ Données de base > Articles > Articles'}
{'Header 1': 'Le champ « Type de délai » peut prendre 6 valeurs :'}
{'Header 1': '2.5 ARTICLES/SITES'}
{'Header 1': '3 EXPLOITATION'}
{'Header 1': 'Impressions > Impressions-groupe > Achats > Analyse'}
{'Header 1': 'Fenêtre de lancement :'}
{'Header 1': 'Site : Site de stockage proposé par défaut (champ obligatoire).'}
{'Header 1': 'ces champs.'}
{'Header 1': 'souhaitée, ne pas renseigner ces champs.'}
{'Header 1': 'sera calculé de façon identique pour tous les fournisseurs de l’article.'}
{'Header 1': 'souh

In [28]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

final_docs = recursive_splitter.split_documents(enhanced_docs)

In [29]:
final_docs

[Document(metadata={'Header 1': 'Guide Utilisateur'}, page_content='Section: Guide Utilisateur\n\nVersion du 22/05/2024  \n1'),
 Document(metadata={'Header 1': 'Descriptif'}, page_content='Section: Descriptif\n\n20/06/2011'),
 Document(metadata={'Header 1': 'Version initiale'}, page_content='Section: Version initiale'),
 Document(metadata={'Header 1': 'Version initiale'}, page_content='19/08/2011  \n27/09/2018  \n2.1, 2.4, 2.5, 3.3  \nPossibilité de calculer le besoin selon un seuil de réappro manuel avec besoin = seuil réappro - Possibilité de calculer le besoin d’un site en prenant en compte les données de plusieurs sites (groupe de site). Mise à jour Guide en Version 11'),
 Document(metadata={'Header 1': 'Version initiale'}, page_content='Possibilité de calculer le besoin selon un seuil de réappro manuel avec besoin = seuil réappro - Possibilité de calculer le besoin d’un site en prenant en compte les données de plusieurs sites (groupe de site). Mise à jour Guide en Version 11  \n22

In [31]:

len(final_docs)

73

In [32]:
final_docs[1]

Document(metadata={'Header 1': 'Descriptif'}, page_content='Section: Descriptif\n\n20/06/2011')

In [33]:
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [34]:
from dotenv import load_dotenv

In [35]:
# Embedding wrapper for LangChain
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3"
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 35332.82it/s]


In [36]:
persist_directory="comnied_chuncking_techniques_embedding"

In [37]:
dimension = 1024
chroma_vdb=Chroma.from_documents(documents=final_docs,
                      embedding=embedding_model,
                      persist_directory=persist_directory,
                      )

In [38]:
chroma_vdb.persist()

C:\Users\tchouarr\AppData\Local\Temp\ipykernel_29548\3808622788.py:1: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  chroma_vdb.persist()


In [39]:
vdb=Chroma(persist_directory=persist_directory,
        embedding_function=embedding_model)

C:\Users\tchouarr\AppData\Local\Temp\ipykernel_29548\35172397.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vdb=Chroma(persist_directory=persist_directory,


In [41]:
retriever=vdb.as_retriever(search_kwargs={"k": 30})

In [42]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [43]:
prompt = ChatPromptTemplate.from_template("""
Tu es un assistant spécialisé dans les documents administratifs français.

Réponds uniquement à partir du contexte fourni.

Contexte:
{context}

Question:
{question}
Si possible de donné le numero de la page,
Si l'information n'est pas présente dans le contexte,
réponds:
"Information non trouvée dans les documents."
""")

In [44]:
chain = prompt | llm | StrOutputParser()

In [45]:
def format_docs(docs):

    formatted = []

    for doc in docs:

        page = doc.metadata.get("page", "Unknown")
        header = doc.metadata.get("header", "Sans titre")
        source = doc.metadata.get("source", "Unknown")

        chunk = f"""
Source: {source}
Page: {page}
Section: {header}

Contenu:
{doc.page_content}
"""

        formatted.append(chunk)

    return "\n\n".join(formatted)

In [46]:
question = "comment calculer le besoin ?"

docs = retriever.invoke(question)

context = format_docs(docs)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

Le calcul du besoin est décrit dans plusieurs sections du contexte fourni. Voici les étapes générales pour calculer le besoin :

1. Calcul du seuil de réappro : 
Seuil = Consommation de la période explorée / nb jours de la période explorée x nb jours conso pour le calcul du seuil (fonction du type de délai).

2. Calcul du besoin brut : 
Besoin brut = Consommation de la période explorée / nb jours de la période explorée x nb jours conso pour le calcul du besoin (fonction du type de délai).

3. Calcul du disponible théorique : 
Dispo = Stock physique - Alloué - Manquants + Encours fournisseurs - Encours services.

4. Calcul du besoin net : 
Besoin net = Besoin brut - disponible théorique.

Ces informations sont présentes dans les sections sans titre, mais malheureusement, les numéros de page ne sont pas fournis. 

Il est important de noter que le calcul du besoin peut varier en fonction du type de délai et d'autres paramètres spécifiques. Il est donc recommandé de consulter les sections 